In [ ]:
!pip install --upgrade patsy

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import os

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy.categorical import C
from patsy.contrasts import Treatment
from patsy import dmatrices
from scipy.stats import zscore
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os

df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

# 分类变量(无序）
cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'Alley','LotShape','LandContour','Utilities','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual',
'GarageCond','PavedDrive','PoolQC','Fence','MiscFeature'
]
#数值变量
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
#参照组字典
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  '1Story',
'RoofStyle':   'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleType':    'WD',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
}

#determine formula
def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

#合并特征
feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("===formula===")
print(formula[:800],"...\n")

#patsy crate X,Y
Y, X = dmatrices(formula, data=df_model, return_type="dataframe")

# ==========【这是你要的新版标准化代码块】==========
X_no_int = X.drop("Intercept", axis=1)

# 区分连续变量项 和 分类哑变量项
num_terms = [c for c in X_no_int.columns if c.split("[")[0] in num_cols]
cat_terms = [c for c in X_no_int.columns if c not in num_terms]

X_num = X_no_int[num_terms].copy()
X_cat = X_no_int[cat_terms].copy()

# 剔除连续变量里面标准差接近0的常数列
stds = X_num.std(axis=0)
keep_num = stds[stds > 1e-10].index
X_num = X_num[keep_num]

# 仅对连续变量做zscore标准化
X_num_std = pd.DataFrame(zscore(X_num.values, axis=0), columns=X_num.columns, index=X_num.index)

# 拼接：标准化后的连续变量 + 不标准化的哑变量
X_std = pd.concat([X_num_std, X_cat], axis=1)
Y_std = pd.Series(zscore(Y[Y_col].values), index=Y.index)
# ================================================

#standard model
X_std = sm.add_constant(X_std, prepend=True)
model_std = sm.OLS(Y_std, X_std)
res_std = model_std.fit()

#raw model
model_raw = sm.OLS(Y, X)
res_raw = model_raw.fit()

#print result
res_df = pd.DataFrame({
"feature_name": res_std.params.index,
"std_beta": res_std.params.values,    # standard coefficient
"t_value": res_std.tvalues.values,
"p_value": res_std.pvalues.values
})
# rank based on p values
res_df = res_df[res_df["feature_name"]!="Intercept"].copy()
res_df = res_df.sort_values("p_value", ascending=True).reset_index(drop=True)

print("=====Standardized full OLS model results (first 20 rows, sorted by p-value ascending) =====")
print(res_df.head(20).to_string(index=False))
print("\n====abstract of non-standard OLS model====")
print(res_raw.summary().tables[0])

#store
save_path = r"..\figures"
#create file
os.makedirs(save_path, exist_ok=True)
file_result = f"{save_path}\\ols_std_all_result.csv"
file_vif = f"{save_path}\\vif_result.csv"

# store
res_df.to_csv(file_result, index=False, encoding="utf-8-sig")
print(f"\nall of standard coefficient have stored in {file_result}")

#VIF evaluate
def calc_vif(Xdf):
    vif_out = pd.DataFrame()
    vif_out["feature"] = Xdf.columns
    vif_out["VIF"] = [variance_inflation_factor(Xdf.values, i) for i in range(Xdf.shape[1])]
    return vif_out

vif_df = calc_vif(X_no_int)
vif_df = vif_df.sort_values("VIF", ascending=False)
print("\n====Top 15 VIF (higher = more severe multicollinearity)====")
print(vif_df.head(15).to_string(index=False))
vif_df.to_csv(file_vif,index=False,encoding="utf-8-sig")
print(f"VIF have stored in {file_vif}")

print(f"\n standardized model R-squared = {res_std.rsquared:.4f}")
print(f"raw model R-squared = {res_raw.rsquared:.4f}, Adj R-square={res_raw.rsquared_adj:.4f}")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy.categorical import C
from patsy.contrasts import Treatment
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os

df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

# 分类变量(无序）
cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'Alley','LotShape','LandContour','Utilities','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual',
'GarageCond','PavedDrive','PoolQC','Fence','MiscFeature'
]
#数值变量
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
#参照组字典
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  '1Story',
'RoofStyle':   'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleType':    'WD',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
}

#determine formula
def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

#合并特征
feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("===formula===")
print(formula[:800],"...\n")

#patsy crate X,Y
Y, X = dmatrices(formula, data=df_model, return_type="dataframe")

# ========== 先不标准化自变量只跑原始OLS ==========
X_no_int = X.drop("Intercept", axis=1)

#raw model
model_raw = sm.OLS(Y, X)
res_raw = model_raw.fit()

# ========== 手动计算标准化beta ==========
beta_raw = res_raw.params
sd_x = X_no_int.std(ddof=1)
sd_y = Y[Y_col].std(ddof=1)

beta_std = beta_raw.copy()
for var_name in X_no_int.columns:
    beta_std[var_name] = beta_raw[var_name] * sd_x[var_name] / sd_y
beta_std["Intercept"] = np.nan

#组装结果表
res_df = pd.DataFrame({
    "feature_name": beta_std.index,
    "std_beta": beta_std.values,
    "raw_beta": beta_raw.values,
    "t_value": res_raw.tvalues.values,
    "p_value": res_raw.pvalues.values
})
res_df = res_df[res_df["feature_name"]!="Intercept"].copy()
res_df = res_df.sort_values("p_value", ascending=True).reset_index(drop=True)

print("=====Standardized beta results (first 20 rows, sorted by p-value ascending) =====")
print(res_df.head(20).to_string(index=False))
print("\n====abstract of OLS model====")
print(res_raw.summary().tables[0])

#store
save_path = r"..\figures"
os.makedirs(save_path, exist_ok=True)
file_result = f"{save_path}\\ols_std_all_result.csv"
file_vif = f"{save_path}\\vif_result.csv"

res_df.to_csv(file_result, index=False, encoding="utf-8-sig")
print(f"\nall of standard coefficient have stored in {file_result}")

#VIF evaluate
def calc_vif(Xdf):
    vif_out = pd.DataFrame()
    vif_out["feature"] = Xdf.columns
    vif_out["VIF"] = [variance_inflation_factor(Xdf.values, i) for i in range(Xdf.shape[1])]
    return vif_out

vif_df = calc_vif(X_no_int)
vif_df = vif_df.sort_values("VIF", ascending=False)
print("\n====Top 15 VIF (higher = more severe multicollinearity)====")
print(vif_df.head(15).to_string(index=False))
vif_df.to_csv(file_vif,index=False,encoding="utf-8-sig")
print(f"VIF have stored in {file_vif}")

print(f"\n raw model R-squared = {res_raw.rsquared:.4f}, Adj R-square={res_raw.rsquared_adj:.4f}")


In [ ]:
df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

# 分类变量(无序）
cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'Alley','LotShape','LandContour','Utilities','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual',
'GarageCond','PavedDrive','PoolQC','Fence','MiscFeature'
]
#数值变量
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
#参照组字典
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  '1Story',
'RoofStyle':   'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleType':    'WD',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
}

def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"

print("feature_cols长度：", len(feature_cols))
print("all_terms长度：", len(all_terms))
print("formula：", formula)


In [ ]:
df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'Alley','LotShape','LandContour','Utilities','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual',
'GarageCond','PavedDrive','PoolQC','Fence','MiscFeature'
]
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  '1Story',
'RoofStyle':   'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleType':    'WD',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
}

def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("formula：", formula[:1000])

Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
X_no_int = X.drop("Intercept", axis=1)
y_arr = Y[Y_col].values
x_arr = X_no_int.values

# 标准化（Ridge必须标准化）
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_arr)

# Ridge回归，alpha自选，alpha越大惩罚越强
ridge = Ridge(alpha=1.0)
ridge.fit(x_scaled, y_arr)

# 标准化系数 std_beta
std_beta = ridge.coef_
# 原始尺度系数 raw_beta
raw_beta = ridge.coef_ / scaler.scale_

res_df = pd.DataFrame({
    "feature_name": X_no_int.columns,
    "std_beta": std_beta,
    "raw_beta": raw_beta
})
res_df = res_df.sort_values("std_beta", key=abs, ascending=False).reset_index(drop=True)

print("\n===== Ridge Regression Result (sorted by absolute std beta) =====")
print(res_df.head(20).to_string(index=False))

# R2
r2 = ridge.score(x_scaled, y_arr)
print(f"\nRidge R-squared = {r2:.4f}")

# 保存
save_path = r"..\figures"
os.makedirs(save_path, exist_ok=True)
out_file = os.path.join(save_path, "ridge_result.csv")
res_df.to_csv(out_file, index=False, encoding="utf-8-sig")
print(f"\nResult saved to {out_file}")

In [ ]:
import pandas as pd
import numpy as np
from patsy import dmatrices
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import os

df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'Alley','LotShape','LandContour','Utilities','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual',
'GarageCond','PavedDrive','PoolQC','Fence','MiscFeature'
]
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  '1Story',
'RoofStyle':   'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleType':    'WD',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
}

# ========== 缺失值填充：防止patsy把样本删光 ==========
# 数值变量：中位数填充
for v in num_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].median())
# 分类变量：众数填充
for v in cat_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("formula ready")

# na_action='raise' 如果你还有NA会直接提醒；na_action='drop' 是删除
Y, X = dmatrices(formula, data=df_model, return_type="dataframe", na_action="drop")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

if X.shape[0]==0:
    raise Exception("全部样本因为缺失被删除，请检查缺失值填充")

X_no_int = X.drop("Intercept", axis=1)
y_arr = Y[Y_col].values
x_arr = X_no_int.values

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_arr)

ridge = Ridge(alpha=1.0)
ridge.fit(x_scaled, y_arr)

std_beta = ridge.coef_
raw_beta = ridge.coef_ / scaler.scale_

res_df = pd.DataFrame({
    "feature_name": X_no_int.columns,
    "std_beta": std_beta,
    "raw_beta": raw_beta
})
res_df = res_df.sort_values("std_beta", key=abs, ascending=False).reset_index(drop=True)

print("\n===== Ridge Result (sorted by |std beta|) =====")
print(res_df.head(20).to_string(index=False))
r2 = ridge.score(x_scaled, y_arr)
print(f"\nRidge R-squared = {r2:.4f}")

save_path = r"..\figures"
os.makedirs(save_path, exist_ok=True)
out_file = os.path.join(save_path, "ridge_result.csv")
res_df.to_csv(out_file, index=False, encoding="utf-8-sig")
print(f"\nSaved to {out_file}")


In [ ]:
import pandas as pd
import numpy as np
from patsy import dmatrices
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import os

df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'Alley','LotShape','LandContour','Utilities','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtFinType1','BsmtFinType2',
'Heating','HeatingQC','Electrical','KitchenQual','Functional','FireplaceQu',
'GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence','MiscFeature'
]
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  '1Story',
'RoofStyle':   'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleType':    'WD',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
}

# ========== 缺失值填充 ==========
for v in num_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].median())
for v in cat_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("formula ready")

# 删掉刚才报错的 na_action 参数，默认就是 NA_action="drop"
Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

if X.shape[0]==0:
    raise Exception("全部样本因为缺失被删除，请检查缺失值填充")

X_no_int = X.drop("Intercept", axis=1)
y_arr = Y[Y_col].values
x_arr = X_no_int.values

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_arr)

ridge = Ridge(alpha=1.0)
ridge.fit(x_scaled, y_arr)

std_beta = ridge.coef_
raw_beta = ridge.coef_ / scaler.scale_

res_df = pd.DataFrame({
    "feature_name": X_no_int.columns,
    "std_beta": std_beta,
    "raw_beta": raw_beta
})
res_df = res_df.sort_values("std_beta", key=abs, ascending=False).reset_index(drop=True)

print("\n===== Ridge Result (sorted by |std beta|) =====")
print(res_df.head(20).to_string(index=False))
r2 = ridge.score(x_scaled, y_arr)
print(f"\nRidge R-squared = {r2:.4f}")

save_path = r"..\figures"
os.makedirs(save_path, exist_ok=True)
out_file = os.path.join(save_path, "ridge_result.csv")
res_df.to_csv(out_file, index=False, encoding="utf-8-sig")
print(f"\nSaved to {out_file}")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os

# ===================== 路径设置 =====================
save_dir = r"..\figures"
os.makedirs(save_dir, exist_ok=True)

df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

# =========【重点：删掉极易引发完全共线性、样本极少的分类变量】=========
drop_cat = ["PoolQC","MiscFeature","Utilities","Alley"]
cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'LotShape','LandContour','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive'
]
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':   'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
'SaleType':    'WD'
}

# 缺失值填充
for v in num_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].median())
for v in cat_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("formula ready")

Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
X_raw = X.drop("Intercept",axis=1)
y_raw = Y[Y_col]

# ========== 先划分训练集、测试集（实验规范！先划分，再标准化）==========
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# 标准化：只用训练集拟合scaler，避免数据泄露
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 给标准化后的X加上截距项，statsmodels OLS
X_train_scaled_const = sm.add_constant(X_train_scaled, prepend=True)
ols_model = sm.OLS(y_train, X_train_scaled_const)
ols_res = ols_model.fit()
print("==== OLS全模型结果（标准化自变量）====")
print(ols_res.summary())

# -------- 1. VIF计算（在【未标准化训练集】上算VIF，VIF不需要标准化）--------
def calc_vif(df_x):
    vif_df = pd.DataFrame()
    vif_df["feature"] = df_x.columns
    vif_df["VIF"] = [variance_inflation_factor(df_x.values,i) for i in range(df_x.shape[1])]
    return vif_df
vif_result = calc_vif(X_train)
print("\n==== VIF ====")
print(vif_result.sort_values("VIF",ascending=False))

# -------- 2. Cook's Distance、学生化残差（强影响点、离群点）--------
influence = ols_res.get_influence()
cooks_d = influence.cooks_distance[0]
stud_resid = influence.studentized_residual
diagnose_df = pd.DataFrame({
    "cooks_d":cooks_d,
    "studentized_resid":stud_resid,
    "fitted_value": ols_res.fittedvalues.values,
    "residual": ols_res.resid.values
})
print("\n==== Cook距离 & 学生化残差（前20条）====")
print(diagnose_df.head(20))

# -------- 3. 残差图（检验正态、同方差，对应PPT假设检验）--------
resid = ols_res.resid
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.scatter(ols_res.fittedvalues, resid)
plt.axhline(y=0,c="red")
plt.xlabel("拟合值")
plt.ylabel("残差")
plt.title("残差同方差检验")
plt.subplot(1,2,2)
plt.hist(resid,bins=30)
plt.title("残差分布")
plt.tight_layout()
plt.savefig(os.path.join(save_dir,"residual_plot.png"), dpi=300)
plt.show()

# -------- 4. Ridge / Lasso 交叉验证选lambda --------
# Ridge
ridge_cv = RidgeCV(alphas=np.logspace(-3,3,100),scoring="neg_mean_squared_error")
ridge_cv.fit(X_train_scaled,y_train)
ridge_pred = ridge_cv.predict(X_test_scaled)
# Lasso
lasso_cv = LassoCV(alphas=np.logspace(-3,3,100),random_state=42)
lasso_cv.fit(X_train_scaled,y_train)
lasso_pred = lasso_cv.predict(X_test_scaled)
# OLS测试集预测
ols_pred = ols_res.predict(sm.add_constant(X_test_scaled,prepend=True))

# -------- 5. 测试集MSE对比（PPT最后一步）--------
mse_ols = mean_squared_error(y_test,ols_pred)
mse_ridge = mean_squared_error(y_test,ridge_pred)
mse_lasso = mean_squared_error(y_test,lasso_pred)
print("\n==== 测试集MSE对比 ====")
print(f"OLS MSE: {mse_ols:.4f}")
print(f"Ridge MSE: {mse_ridge:.4f}")
print(f"Lasso MSE: {mse_lasso:.4f}")

# 汇总MSE表格
mse_df = pd.DataFrame({
    "Model":["OLS","Ridge","Lasso"],
    "Test_MSE":[mse_ols, mse_ridge, mse_lasso]
})

# 系数汇总表
coef_df = pd.DataFrame({
    "feature": ["Intercept"]+list(X_raw.columns),
    "ols_std_coef": ols_res.params.values,
    "p_value": ols_res.pvalues.values
})

# ===================== 全部结果导出csv到目标文件夹 =====================
coef_df.to_csv(os.path.join(save_dir,"ols_std_coef_pvalue.csv"),index=False,encoding="utf-8-sig")
vif_result.to_csv(os.path.join(save_dir,"vif_result.csv"),index=False,encoding="utf-8-sig")
diagnose_df.to_csv(os.path.join(save_dir,"cook_student_resid.csv"),index=False,encoding="utf-8-sig")
mse_df.to_csv(os.path.join(save_dir,"model_mse_compare.csv"),index=False,encoding="utf-8-sig")

# 保存OLS完整summary文本
with open(os.path.join(save_dir,"ols_summary.txt"),"w",encoding="utf-8") as f:
    f.write(ols_res.summary().as_text())

print(f"\n✅ 所有结果、残差图已经保存至：{save_dir}")
print("文件清单：")
print(" - ols_std_coef_pvalue.csv  标准化系数 + p值")
print(" - vif_result.csv           VIF值")
print(" - cook_student_resid.csv   Cook距离、学生化残差")
print(" - model_mse_compare.csv    三个模型测试集MSE")
print(" - ols_summary.txt          OLS完整输出结果")
print(" - residual_plot.png        残差诊断图")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import warnings

# ===================== 路径设置 =====================
save_dir = r"..\figures"
print(f"目标保存路径：{save_dir}")

# 检查父文件夹是否存在
parent_path = r"..\.."
if os.path.exists(parent_path):
    print("父文件夹 5001 存在")
else:
    print("【重要】桌面没有 5001 文件夹！请先手动在桌面新建文件夹，命名为 5001")
os.makedirs(save_dir, exist_ok=True)
if os.path.exists(save_dir):
    print("result文件夹已经准备好")
else:
    print("result文件夹创建失败，权限问题！")

df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

# =========【重点：删掉极易引发完全共线性、样本极少的分类变量】=========
drop_cat = ["PoolQC","MiscFeature","Utilities","Alley"]
cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'LotShape','LandContour','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive'
]
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
'SaleType':    'WD'
}

# 缺失值填充
for v in num_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].median())
for v in cat_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("formula ready")

Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
X_raw = X.drop("Intercept",axis=1)
y_raw = Y[Y_col]

# ========== 先划分训练集、测试集（实验规范！先划分，再标准化）==========
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# 标准化：只用训练集拟合scaler，避免数据泄露
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 给标准化后的X加上截距项，statsmodels OLS
X_train_scaled_const = sm.add_constant(X_train_scaled, prepend=True)
ols_model = sm.OLS(y_train, X_train_scaled_const)
ols_res = ols_model.fit()
print("==== OLS全模型结果（标准化自变量）====")
print(ols_res.summary())

# -------- VIF：捕获除零警告，无穷VIF标记为Inf --------
def calc_vif(df_x):
    vif_df = pd.DataFrame()
    vif_df["feature"] = df_x.columns
    vif_list = []
    for i in range(df_x.shape[1]):
        try:
            v = variance_inflation_factor(df_x.values, i)
            if np.isinf(v) or np.isnan(v):
                vif_list.append(np.inf)
            else:
                vif_list.append(v)
        except Exception as e:
            print(f"VIF计算报错 {i}: {e}")
            vif_list.append(np.inf)
    vif_df["VIF"] = vif_list
    return vif_df

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    vif_result = calc_vif(X_train)
print("\n==== VIF ====")
print(vif_result.sort_values("VIF",ascending=False))

# -------- Cook's Distance、学生化残差 --------
influence = ols_res.get_influence()
cooks_d = influence.cooks_distance[0]
stud_resid = influence.studentized_residual
diagnose_df = pd.DataFrame({
    "cooks_d":cooks_d,
    "studentized_resid":stud_resid,
    "fitted_value": ols_res.fittedvalues.values,
    "residual": ols_res.resid.values
})
print("\n==== Cook距离 & 学生化残差（前20条）====")
print(diagnose_df.head(20))

# -------- 残差图 --------
resid = ols_res.resid
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.scatter(ols_res.fittedvalues, resid)
plt.axhline(y=0,c="red")
plt.xlabel("拟合值")
plt.ylabel("残差")
plt.title("残差同方差检验")
plt.subplot(1,2,2)
plt.hist(resid,bins=30)
plt.title("残差分布")
plt.tight_layout()
pic_path = os.path.join(save_dir,"residual_plot.png")
plt.savefig(pic_path, dpi=300)
plt.show()
print(f"✅ 残差图片已写入：{pic_path}")

# -------- Ridge / Lasso 交叉验证 --------
ridge_cv = RidgeCV(alphas=np.logspace(-3,3,100),scoring="neg_mean_squared_error")
ridge_cv.fit(X_train_scaled,y_train)
ridge_pred = ridge_cv.predict(X_test_scaled)

lasso_cv = LassoCV(alphas=np.logspace(-3,3,100),random_state=42)
lasso_cv.fit(X_train_scaled,y_train)
lasso_pred = lasso_cv.predict(X_test_scaled)

ols_pred = ols_res.predict(sm.add_constant(X_test_scaled,prepend=True))

# -------- 测试集MSE对比 --------
mse_ols = mean_squared_error(y_test,ols_pred)
mse_ridge = mean_squared_error(y_test,ridge_pred)
mse_lasso = mean_squared_error(y_test,lasso_pred)
print("\n==== 测试集MSE对比 ====")
print(f"OLS MSE: {mse_ols:.4f}")
print(f"Ridge MSE: {mse_ridge:.4f}")
print(f"Lasso MSE: {mse_lasso:.4f}")

mse_df = pd.DataFrame({
    "Model":["OLS","Ridge","Lasso"],
    "Test_MSE":[mse_ols, mse_ridge, mse_lasso]
})

coef_df = pd.DataFrame({
    "feature": ["Intercept"]+list(X_raw.columns),
    "ols_std_coef": ols_res.params.values,
    "p_value": ols_res.pvalues.values
})

# ===================== 导出文件（每写完一个立刻打印提示！） =====================
f1 = os.path.join(save_dir,"ols_std_coef_pvalue.csv")
coef_df.to_csv(f1,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f1}")

f2 = os.path.join(save_dir,"vif_result.csv")
vif_result.to_csv(f2,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f2}")

f3 = os.path.join(save_dir,"cook_student_resid.csv")
diagnose_df.to_csv(f3,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f3}")

f4 = os.path.join(save_dir,"model_mse_compare.csv")
mse_df.to_csv(f4,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f4}")

f5 = os.path.join(save_dir,"ols_summary.txt")
with open(f5,"w",encoding="utf-8") as f:
    f.write(ols_res.summary().as_text())
print(f"✅ 已保存：{f5}")

print(f"\n全部文件写入完成！去文件夹查看：\n{save_dir}")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import warnings

# ===================== 路径设置 =====================
save_dir = r"..\figures"
print(f"目标保存路径：{save_dir}")

parent_path = r"..\.."
if os.path.exists(parent_path):
    print("✅ 父文件夹 5001 存在")
else:
    print("❌ 【重要】桌面没有 5001 文件夹！请先手动在桌面新建文件夹，命名为 5001")
os.makedirs(save_dir, exist_ok=True)
if os.path.exists(save_dir):
    print("✅ result文件夹已经准备好")
else:
    print("❌ result文件夹创建失败，权限问题！")

df_model=pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch":"ThreeSsnPorch"})
Y_col='SalePriceLog'

# =========【重点：删掉极易引发完全共线性、样本极少的分类变量】=========
drop_cat = ["PoolQC","MiscFeature","Utilities","Alley"]
cat_cols = [
'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
'LotShape','LandContour','LotConfig','LandSlope',
'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure',
'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive'
]
num_cols = [
'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
'MoSold','YrSold'
]
reference_levels = {
'MSZoning':    'RL',
'Neighborhood':'NAmes',
'BldgType':    '1Fam',
'HouseStyle':  'Gable',
'Foundation':  'PConc',
'GarageType':  'Attchd',
'SaleCondition':'Normal',
'CentralAir':  'Y',
'Street':      'Pave',
'SaleType':    'WD'
}

# 缺失值填充
for v in num_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].median())
for v in cat_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("formula ready")

Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
X_raw = X.drop("Intercept",axis=1)
y_raw = Y[Y_col]

# ========== 先划分训练集、测试集（实验规范！先划分，再标准化）==========
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# 标准化：只用训练集拟合scaler，避免数据泄露
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 给标准化后的X加上截距项，statsmodels OLS
X_train_scaled_const = sm.add_constant(X_train_scaled, prepend=True)
ols_model = sm.OLS(y_train, X_train_scaled_const)
ols_res = ols_model.fit()
print("==== OLS全模型结果（标准化自变量）====")
print(ols_res.summary())

# -------- VIF：捕获除零警告，无穷VIF标记为Inf --------
def calc_vif(df_x):
    vif_df = pd.DataFrame()
    vif_df["feature"] = df_x.columns
    vif_list = []
    for i in range(df_x.shape[1]):
        try:
            v = variance_inflation_factor(df_x.values, i)
            if np.isinf(v) or np.isnan(v):
                vif_list.append(np.inf)
            else:
                vif_list.append(v)
        except Exception as e:
            print(f"VIF计算报错 {i}: {e}")
            vif_list.append(np.inf)
    vif_df["VIF"] = vif_list
    return vif_df

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    vif_result = calc_vif(X_train)
print("\n==== VIF ====")
print(vif_result.sort_values("VIF",ascending=False))

# -------- Cook's Distance、学生化残差【这里修复！！】 --------
influence = ols_res.get_influence()
cooks_d = influence.cooks_distance[0]
# 改成 resid_studentized_external：外部学生化残差（课程作业推荐这个）
stud_resid = influence.resid_studentized_external
diagnose_df = pd.DataFrame({
    "cooks_d":cooks_d,
    "studentized_resid":stud_resid,
    "fitted_value": ols_res.fittedvalues.values,
    "residual": ols_res.resid.values
})
print("\n==== Cook距离 & 学生化残差（前20条）====")
print(diagnose_df.head(20))

# -------- 残差图 --------
resid = ols_res.resid
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.scatter(ols_res.fittedvalues, resid)
plt.axhline(y=0,c="red")
plt.xlabel("拟合值")
plt.ylabel("残差")
plt.title("残差同方差检验")
plt.subplot(1,2,2)
plt.hist(resid,bins=30)
plt.title("残差分布")
plt.tight_layout()
pic_path = os.path.join(save_dir,"residual_plot.png")
plt.savefig(pic_path, dpi=300)
plt.show()
print(f"✅ 残差图片已写入：{pic_path}")

# -------- Ridge / Lasso 交叉验证 --------
ridge_cv = RidgeCV(alphas=np.logspace(-3,3,100),scoring="neg_mean_squared_error")
ridge_cv.fit(X_train_scaled,y_train)
ridge_pred = ridge_cv.predict(X_test_scaled)

lasso_cv = LassoCV(alphas=np.logspace(-3,3,100),random_state=42)
lasso_cv.fit(X_train_scaled,y_train)
lasso_pred = lasso_cv.predict(X_test_scaled)

ols_pred = ols_res.predict(sm.add_constant(X_test_scaled,prepend=True))

# -------- 测试集MSE对比 --------
mse_ols = mean_squared_error(y_test,ols_pred)
mse_ridge = mean_squared_error(y_test,ridge_pred)
mse_lasso = mean_squared_error(y_test,lasso_pred)
print("\n==== 测试集MSE对比 ====")
print(f"OLS MSE: {mse_ols:.4f}")
print(f"Ridge MSE: {mse_ridge:.4f}")
print(f"Lasso MSE: {mse_lasso:.4f}")

mse_df = pd.DataFrame({
    "Model":["OLS","Ridge","Lasso"],
    "Test_MSE":[mse_ols, mse_ridge, mse_lasso]
})

coef_df = pd.DataFrame({
    "feature": ["Intercept"]+list(X_raw.columns),
    "ols_std_coef": ols_res.params.values,
    "p_value": ols_res.pvalues.values
})

# ===================== 导出文件（每写完一个立刻打印提示！） =====================
f1 = os.path.join(save_dir,"ols_std_coef_pvalue.csv")
coef_df.to_csv(f1,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f1}")

f2 = os.path.join(save_dir,"vif_result.csv")
vif_result.to_csv(f2,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f2}")

f3 = os.path.join(save_dir,"cook_student_resid.csv")
diagnose_df.to_csv(f3,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f3}")

f4 = os.path.join(save_dir,"model_mse_compare.csv")
mse_df.to_csv(f4,index=False,encoding="utf-8-sig")
print(f"✅ 已保存：{f4}")

f5 = os.path.join(save_dir,"ols_summary.txt")
with open(f5,"w",encoding="utf-8") as f:
    f.write(ols_res.summary().as_text())
print(f"✅ 已保存：{f5}")

print(f"\n🎉 全部文件写入完成！去下面这个文件夹查看：\n{save_dir}")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import warnings

# ===================== 1. 路径设置 =====================
save_dir = r"..\figures"
print(f"目标保存路径：{save_dir}")

parent_path = r"..\.."
if os.path.exists(parent_path):
    print("✅ 父文件夹 5001 存在")
else:
    print("❌ 【重要】桌面没有 5001 文件夹！请先手动在桌面新建文件夹，命名为 5001")

os.makedirs(save_dir, exist_ok=True)
if os.path.exists(save_dir):
    print("✅ result文件夹已经准备好")
else:
    print("❌ result文件夹创建失败，权限问题！")

# ===================== 2. 数据读取与特征定义 =====================
df_model = pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch": "ThreeSsnPorch"})
Y_col = 'SalePriceLog'

drop_cat = ["PoolQC", "MiscFeature", "Utilities", "Alley"]

cat_cols = [
    'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
    'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
    'LotShape','LandContour','LotConfig','LandSlope',
    'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
    'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtExposure',
    'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
    'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive'
]

num_cols = [
    'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
    'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
    'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
    'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
    'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
    'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
    'MoSold','YrSold'
]

reference_levels = {
    'MSZoning':    'RL',
    'Neighborhood':'NAmes',
    'BldgType':    '1Fam',
    'HouseStyle':  'Gable',
    'Foundation':  'PConc',
    'GarageType':  'Attchd',
    'SaleCondition':'Normal',
    'CentralAir':  'Y',
    'Street':      'Pave',
    'SaleType':    'WD'
}

# 缺失值填充
for v in num_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].median())
for v in cat_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

# ===================== 3. 构建公式 =====================
def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("✅ 回归公式已生成")

# 生成设计矩阵
Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
X_raw = X.drop("Intercept", axis=1)
y_raw = Y[Y_col]

# ===================== 4. 划分数据集与标准化 =====================
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

scaler = StandardScaler()
# 【优化】将标准化后的 numpy 数组转回 DataFrame，保留特征名
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# 给标准化训练集加截距，拟合 OLS
X_train_scaled_const = sm.add_constant(X_train_scaled, prepend=True)
ols_model = sm.OLS(y_train, X_train_scaled_const)
ols_res = ols_model.fit()

print("\n==== OLS全模型结果（标准化自变量）====")
print(ols_res.summary())

# ===================== 5. 诊断：VIF =====================
def calc_vif(df_x):
    vif_df = pd.DataFrame()
    vif_df["feature"] = df_x.columns
    vif_list = []
    for i in range(df_x.shape[1]):
        try:
            v = variance_inflation_factor(df_x.values, i)
            if np.isinf(v) or np.isnan(v):
                vif_list.append(np.inf)
            else:
                vif_list.append(v)
        except Exception as e:
            print(f"VIF计算报错 {df_x.columns[i]}: {e}")
            vif_list.append(np.inf)
    vif_df["VIF"] = vif_list
    return vif_df

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    vif_result = calc_vif(X_train)  # 用未标准化的数据算VIF，不影响结果

print("\n==== VIF (前15个最高值) ====")
print(vif_result.sort_values("VIF", ascending=False).head(15))

# ===================== 6. 诊断：Cook's Distance 与 学生化残差 =====================
influence = ols_res.get_influence()
hii = influence.hat_matrix_diag
mask = hii < 0.995  # 过滤极端高杠杆点
cooks_d = influence.cooks_distance[0]
stud_resid = influence.resid_studentized_external

cooks_d_clean = np.where(mask, cooks_d, np.nan)
stud_resid_clean = np.where(mask, stud_resid, np.nan)

diagnose_df = pd.DataFrame({
    "hii": hii,
    "cooks_d": cooks_d_clean,
    "studentized_resid": stud_resid_clean,
    "fitted_value": ols_res.fittedvalues.values,
    "residual": ols_res.resid.values
})
print("\n==== Cook距离 & 学生化残差（前5条，完整数据将保存至CSV）====")
print(diagnose_df.head(5))

# ===================== 7. 诊断图：残差图 =====================
resid = ols_res.resid
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(ols_res.fittedvalues, resid, alpha=0.5)
plt.axhline(y=0, c="red")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted (Homoscedasticity)")

plt.subplot(1, 2, 2)
plt.hist(resid, bins=30, edgecolor='black')
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()

pic_path = os.path.join(save_dir, "residual_plot.png")
plt.savefig(pic_path, dpi=300)
plt.close()  # 关闭画布，释放内存
print(f"✅ 残差图片已写入：{pic_path}")

# ===================== 8. 正则化模型：Ridge / Lasso =====================
# Ridge
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), scoring="neg_mean_squared_error")
ridge_cv.fit(X_train_scaled, y_train)
ridge_pred = ridge_cv.predict(X_test_scaled)

# Lasso
lasso_cv = LassoCV(alphas=np.logspace(-3, 3, 100), random_state=42)
lasso_cv.fit(X_train_scaled, y_train)
lasso_pred = lasso_cv.predict(X_test_scaled)

# OLS 预测测试集（修复：直接用相同的StandardScaler处理后的X_test）
X_test_scaled_const = sm.add_constant(X_test_scaled, prepend=True)
ols_pred = ols_res.predict(X_test_scaled_const)

# ===================== 9. 模型评估：MSE对比 =====================
mse_ols = mean_squared_error(y_test, ols_pred)
mse_ridge = mean_squared_error(y_test, ridge_pred)
mse_lasso = mean_squared_error(y_test, lasso_pred)

print("\n==== 测试集 MSE 对比 ====")
print(f"OLS MSE:   {mse_ols:.6f}")
print(f"Ridge MSE: {mse_ridge:.6f}")
print(f"Lasso MSE: {mse_lasso:.6f}")

mse_df = pd.DataFrame({
    "Model": ["OLS", "Ridge", "Lasso"],
    "Test_MSE": [mse_ols, mse_ridge, mse_lasso]
})

# ===================== 10. 汇总结果与导出 =====================
coef_df = pd.DataFrame({
    "feature": ols_res.params.index,
    "ols_std_coef": ols_res.params.values,
    "p_value": ols_res.pvalues.values
})

# 导出 1: 系数与 P 值
f1 = os.path.join(save_dir, "ols_std_coef_pvalue.csv")
coef_df.to_csv(f1, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f1}")

# 导出 2: VIF
f2 = os.path.join(save_dir, "vif_result.csv")
vif_result.to_csv(f2, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f2}")

# 导出 3: 诊断数据 (Cook距离等)
f3 = os.path.join(save_dir, "cook_student_resid.csv")
diagnose_df.to_csv(f3, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f3}")

# 导出 4: MSE 对比
f4 = os.path.join(save_dir, "model_mse_compare.csv")
mse_df.to_csv(f4, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f4}")

# 导出 5: 文本版 Summary
f5 = os.path.join(save_dir, "ols_summary.txt")
with open(f5, "w", encoding="utf-8") as f:
    f.write(ols_res.summary().as_text())
print(f"✅ 已保存：{f5}")

print(f"\n🎉 全部文件写入完成！去下面这个文件夹查看：\n{save_dir}")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import warnings

# ===================== 1. 路径设置 =====================
# 修改为你的目标文件夹
save_dir = r"..\tables"
print(f"目标保存路径：{save_dir}")

parent_path = r"..\.."
if os.path.exists(parent_path):
    print("✅ 父文件夹 5001 存在")
else:
    print("❌ 【重要】桌面没有 5001 文件夹！请先手动在桌面新建文件夹，命名为 5001")

os.makedirs(save_dir, exist_ok=True)
if os.path.exists(save_dir):
    print("✅ outcome 文件夹已经准备好")
else:
    print("❌ outcome 文件夹创建失败，权限问题！")

# ===================== 2. 数据读取与特征定义 =====================
df_model = pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch": "ThreeSsnPorch"})
Y_col = 'SalePriceLog'

drop_cat = ["PoolQC", "MiscFeature", "Utilities", "Alley"]

cat_cols = [
    'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
    'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
    'LotShape','LandContour','LotConfig','LandSlope',
    'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
    'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtExposure',
    'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
    'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive'
]

num_cols = [
    'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
    'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
    'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
    'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
    'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
    'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
    'MoSold','YrSold'
]

reference_levels = {
    'MSZoning':    'RL',
    'Neighborhood':'NAmes',
    'BldgType':    '1Fam',
    'HouseStyle':  'Gable',
    'Foundation':  'PConc',
    'GarageType':  'Attchd',
    'SaleCondition':'Normal',
    'CentralAir':  'Y',
    'Street':      'Pave',
    'SaleType':    'WD'
}

# 缺失值填充
for v in num_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].median())
for v in cat_cols:
    if v in df_model.columns:
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

# ===================== 3. 构建公式 =====================
def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"
print("✅ 回归公式已生成")

# 生成设计矩阵
Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
X_raw = X.drop("Intercept", axis=1)
y_raw = Y[Y_col]

# ===================== 4. 划分数据集与标准化 =====================
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# 【核心修复1】清理在全量数据中只有一个取值的常数列，防止标准化除以0产生 NaN
non_constant_cols = X_train.columns[X_train.std() > 0]
X_train = X_train[non_constant_cols]
X_test = X_test[non_constant_cols]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# 给标准化训练集加截距，拟合 OLS
X_train_scaled_const = sm.add_constant(X_train_scaled, prepend=True)
ols_model = sm.OLS(y_train, X_train_scaled_const)
ols_res = ols_model.fit()

print("\n==== OLS全模型结果（标准化自变量）====")
print(ols_res.summary())

# ===================== 5. 诊断：VIF =====================
def calc_vif(df_x):
    vif_df = pd.DataFrame()
    vif_df["feature"] = df_x.columns
    vif_list = []
    for i in range(df_x.shape[1]):
        try:
            v = variance_inflation_factor(df_x.values, i)
            if np.isinf(v) or np.isnan(v):
                vif_list.append(np.inf)
            else:
                vif_list.append(v)
        except Exception as e:
            print(f"VIF计算报错 {df_x.columns[i]}: {e}")
            vif_list.append(np.inf)
    vif_df["VIF"] = vif_list
    return vif_df

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    vif_result = calc_vif(X_train)  # 用未标准化的数据算VIF

print("\n==== VIF (前15个最高值) ====")
print(vif_result.sort_values("VIF", ascending=False).head(15))

# ===================== 6. 诊断：Cook's Distance 与 学生化残差 =====================
influence = ols_res.get_influence()
hii = influence.hat_matrix_diag
mask = hii < 0.995  # 过滤极端高杠杆点
cooks_d = influence.cooks_distance[0]
stud_resid = influence.resid_studentized_external

cooks_d_clean = np.where(mask, cooks_d, np.nan)
stud_resid_clean = np.where(mask, stud_resid, np.nan)

diagnose_df = pd.DataFrame({
    "hii": hii,
    "cooks_d": cooks_d_clean,
    "studentized_resid": stud_resid_clean,
    "fitted_value": ols_res.fittedvalues.values,
    "residual": ols_res.resid.values
})
print("\n==== Cook距离 & 学生化残差（前5条）====")
print(diagnose_df.head(5))

# ===================== 7. 诊断图：残差图 =====================
resid = ols_res.resid
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(ols_res.fittedvalues, resid, alpha=0.5)
plt.axhline(y=0, c="red")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted (Homoscedasticity)")

plt.subplot(1, 2, 2)
plt.hist(resid, bins=30, edgecolor='black')
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()

pic_path = os.path.join(save_dir, "residual_plot.png")
plt.savefig(pic_path, dpi=300)
plt.close()  # 关闭画布，释放内存
print(f"✅ 残差图片已写入：{pic_path}")

# ===================== 8. 正则化模型：Ridge / Lasso =====================
# Ridge
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), scoring="neg_mean_squared_error")
ridge_cv.fit(X_train_scaled, y_train)
ridge_pred = ridge_cv.predict(X_test_scaled)

# Lasso
lasso_cv = LassoCV(alphas=np.logspace(-3, 3, 100), random_state=42, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)
lasso_pred = lasso_cv.predict(X_test_scaled)

# OLS 预测测试集
# 【核心修复2】强制让测试集的列名和顺序与训练集一模一样，彻底消灭维度报错
X_test_scaled = X_test_scaled[X_train_scaled.columns]
X_test_scaled_const = sm.add_constant(X_test_scaled, prepend=True)
# 再按训练集的列顺序强制对齐一次
X_test_scaled_const = X_test_scaled_const[X_train_scaled_const.columns]

ols_pred = ols_res.predict(X_test_scaled_const)

# ===================== 9. 模型评估：MSE对比 =====================
mse_ols = mean_squared_error(y_test, ols_pred)
mse_ridge = mean_squared_error(y_test, ridge_pred)
mse_lasso = mean_squared_error(y_test, lasso_pred)

print("\n==== 测试集 MSE 对比 ====")
print(f"OLS MSE:   {mse_ols:.6f}")
print(f"Ridge MSE: {mse_ridge:.6f}")
print(f"Lasso MSE: {mse_lasso:.6f}")

mse_df = pd.DataFrame({
    "Model": ["OLS", "Ridge", "Lasso"],
    "Test_MSE": [mse_ols, mse_ridge, mse_lasso]
})

# ===================== 10. 汇总结果与导出 =====================
coef_df = pd.DataFrame({
    "feature": ols_res.params.index,
    "ols_std_coef": ols_res.params.values,
    "p_value": ols_res.pvalues.values
})

# 导出 1: 系数与 P 值
f1 = os.path.join(save_dir, "ols_std_coef_pvalue.csv")
coef_df.to_csv(f1, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f1}")

# 导出 2: VIF
f2 = os.path.join(save_dir, "vif_result.csv")
vif_result.to_csv(f2, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f2}")

# 导出 3: 诊断数据 (Cook距离等)
f3 = os.path.join(save_dir, "cook_student_resid.csv")
diagnose_df.to_csv(f3, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f3}")

# 导出 4: MSE 对比
f4 = os.path.join(save_dir, "model_mse_compare.csv")
mse_df.to_csv(f4, index=False, encoding="utf-8-sig")
print(f"✅ 已保存：{f4}")

# 导出 5: 文本版 Summary
f5 = os.path.join(save_dir, "ols_summary.txt")
with open(f5, "w", encoding="utf-8") as f:
    f.write(ols_res.summary().as_text())
print(f"✅ 已保存：{f5}")

print(f"\n🎉 全部文件写入完成！去下面这个文件夹查看：\n{save_dir}")

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import warnings

# 屏蔽无关警告（如Lasso收敛警告）
warnings.filterwarnings("ignore")

# ===================== 1. 路径设置 =====================
save_dir = r"..\tables"
print(f"目标保存路径：{save_dir}")
os.makedirs(save_dir, exist_ok=True)

# ===================== 2. 数据读取与特征定义 =====================
df_model = pd.read_csv(r'..\..\part1\data\train_clean.csv')
df_model = df_model.rename(columns={"3SsnPorch": "ThreeSsnPorch"})
Y_col = 'SalePriceLog'

cat_cols = [
    'MSZoning','Neighborhood','BldgType','HouseStyle','RoofStyle',
    'Foundation','GarageType','SaleType','SaleCondition','CentralAir','Street',
    'LotShape','LandContour','LotConfig','LandSlope',
    'Condition1','Condition2','RoofMatl','Exterior1st','Exterior2nd',
    'MasVnrType','ExterQual','ExterCond','BsmtQual','BsmtExposure',
    'BsmtFinType1','BsmtFinType2','Heating','HeatingQC','Electrical',
    'KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive'
]

num_cols = [
    'MSSubClass','LotFrontage','LotArea','OverallQual','OverallCond',
    'YearBuilt','YearRemodAdd','MasVnrArea','TotalBsmtSF','GrLivArea',
    'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath',
    'BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd','Fireplaces',
    'GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF',
    'EnclosedPorch','ThreeSsnPorch','ScreenPorch','PoolArea','MiscVal',
    'MoSold','YrSold'
]

reference_levels = {
    'MSZoning':    'RL', 'Neighborhood':'NAmes', 'BldgType':    '1Fam',
    'HouseStyle':  'Gable', 'Foundation':  'PConc', 'GarageType':  'Attchd',
    'SaleCondition':'Normal', 'CentralAir':  'Y', 'Street':      'Pave', 'SaleType':    'WD'
}

# 缺失值填充
for v in num_cols:
    if v in df_model.columns: 
        df_model[v] = df_model[v].fillna(df_model[v].median())
for v in cat_cols:
    if v in df_model.columns: 
        df_model[v] = df_model[v].fillna(df_model[v].mode()[0])

# ===================== 3. 构建公式 =====================
def term_of(c):
    if c in cat_cols:
        ref = reference_levels.get(c)
        if ref is not None and ref in df_model[c].unique():
            return f"C({c}, Treatment(reference='{ref}'))"
        else:
            return f'C({c})'
    else:
        return c

feature_cols = cat_cols + num_cols
all_terms = [term_of(col) for col in feature_cols]
formula = f"{Y_col} ~ {' + '.join(all_terms)}"

# 生成设计矩阵
Y, X = dmatrices(formula, data=df_model, return_type="dataframe")
X_raw = X.drop("Intercept", axis=1)
y_raw = Y[Y_col]

# ===================== 4. 划分数据集与标准化 =====================
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# 清理常数列（防止标准化除以0）
non_constant_cols = X_train.columns[X_train.std() > 0]
X_train = X_train[non_constant_cols]
X_test = X_test[non_constant_cols]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# 清理所有的 NaN 和 Inf
X_train_scaled = X_train_scaled.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_scaled = X_test_scaled.replace([np.inf, -np.inf], np.nan).fillna(0)

# ===================== 5. OLS 模型拟合 =====================
#给训练集插入截距列
X_train_scaled_const = X_train_scaled.copy()
X_train_scaled_const.insert(0, 'const', 1.0)

ols_model = sm.OLS(y_train, X_train_scaled_const)
ols_res = ols_model.fit()

print("==== OLS全模型结果（标准化自变量）====")
print(ols_res.summary())

# ===================== 6. 诊断：VIF =====================
def calc_vif(df_x):
    vif_df = pd.DataFrame()
    vif_df["feature"] = df_x.columns
    vif_list = []
    for i in range(df_x.shape[1]):
        try:
            v = variance_inflation_factor(df_x.values, i)
            if np.isinf(v) or np.isnan(v): 
                vif_list.append(np.inf)
            else: 
                vif_list.append(v)
        except Exception:
            vif_list.append(np.inf)
    vif_df["VIF"] = vif_list
    return vif_df

vif_result = calc_vif(X_train) 
print("\n==== VIF (前15个最高值) ====")
print(vif_result.sort_values("VIF", ascending=False).head(15))

# ===================== 7. 诊断：Cook's Distance 与 学生化残差 =====================
influence = ols_res.get_influence()
hii = influence.hat_matrix_diag
mask = hii < 0.995  # 过滤极端高杠杆点
cooks_d = influence.cooks_distance[0]
stud_resid = influence.resid_studentized_external

cooks_d_clean = np.where(mask, cooks_d, np.nan)
stud_resid_clean = np.where(mask, stud_resid, np.nan)

diagnose_df = pd.DataFrame({
    "hii": hii,
    "cooks_d": cooks_d_clean,
    "studentized_resid": stud_resid_clean,
    "fitted_value": ols_res.fittedvalues.values,
    "residual": ols_res.resid.values
})
print("\n==== Cook距离 & 学生化残差（前5条）====")
print(diagnose_df.head(5))

# ===================== 8. 诊断图：残差图 =====================
resid = ols_res.resid
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(ols_res.fittedvalues, resid, alpha=0.5)
plt.axhline(y=0, c="red")
plt.title("Residuals vs Fitted")
plt.subplot(1, 2, 2)
plt.hist(resid, bins=30, edgecolor='black')
plt.title("Residual Distribution")
plt.tight_layout()

pic_path = os.path.join(save_dir, "residual_plot.png")
plt.savefig(pic_path, dpi=300)
plt.close() 
print(f"残差图片已写入：{pic_path}")

# ===================== 9. 正则化模型与预测 =====================
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), scoring="neg_mean_squared_error")
ridge_cv.fit(X_train_scaled, y_train)
ridge_pred = ridge_cv.predict(X_test_scaled)

lasso_cv = LassoCV(alphas=np.logspace(-3, 3, 100), random_state=42, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)
lasso_pred = lasso_cv.predict(X_test_scaled)

#手动给测试集插入截距，强制对齐列名
X_test_scaled_const = X_test_scaled.copy()
X_test_scaled_const.insert(0, 'const', 1.0)
X_test_scaled_const = X_test_scaled_const[X_train_scaled_const.columns]

ols_pred = ols_res.predict(X_test_scaled_const)

# ===================== 10. 模型评估：MSE对比 =====================
mse_ols = mean_squared_error(y_test, ols_pred)
mse_ridge = mean_squared_error(y_test, ridge_pred)
mse_lasso = mean_squared_error(y_test, lasso_pred)

print("\n==== 测试集 MSE 对比 ====")
print(f"OLS MSE:   {mse_ols:.6f}")
print(f"Ridge MSE: {mse_ridge:.6f}")
print(f"Lasso MSE: {mse_lasso:.6f}")

mse_df = pd.DataFrame({"Model": ["OLS", "Ridge", "Lasso"], "Test_MSE": [mse_ols, mse_ridge, mse_lasso]})

coef_df = pd.DataFrame({
    "feature": ols_res.params.index,
    "ols_std_coef": ols_res.params.values,
    "p_value": ols_res.pvalues.values
})

# ===================== 11. 汇总结果与导出 =====================
print("\n--- 开始保存文件 ---")

f1 = os.path.join(save_dir, "ols_std_coef_pvalue.csv")
coef_df.to_csv(f1, index=False, encoding="utf-8-sig")

f2 = os.path.join(save_dir, "vif_result.csv")
vif_result.to_csv(f2, index=False, encoding="utf-8-sig")

f3 = os.path.join(save_dir, "cook_student_resid.csv")
diagnose_df.to_csv(f3, index=False, encoding="utf-8-sig")

f4 = os.path.join(save_dir, "model_mse_compare.csv")
mse_df.to_csv(f4, index=False, encoding="utf-8-sig")

f5 = os.path.join(save_dir, "ols_summary.txt")
with open(f5, "w", encoding="utf-8") as f:
    f.write(ols_res.summary().as_text())

#校验
for file_path in [f1, f2, f3, f4, f5]:
    if os.path.exists(file_path):
        print(f"成功保存：{os.path.basename(file_path)}")
    else:
        print(f"保存失败：{os.path.basename(file_path)}")

print(f"\n全部执行完毕去文件夹查看：\n{save_dir}")

目标保存路径：C:\Users\HONOR\Desktop\5001\outcome
==== OLS全模型结果（标准化自变量）====
                            OLS Regression Results                            
Dep. Variable:           SalePriceLog   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.929
Method:                 Least Squares   F-statistic:                     69.29
Date:                Wed, 18 Nov 2026   Prob (F-statistic):               0.00
Time:                        23:38:54   Log-Likelihood:                 1105.2
No. Observations:                1168   AIC:                            -1764.
Df Residuals:                     945   BIC:                            -635.3
Df Model:                         222                                         
Covariance Type:            nonrobust                                         
                                                                 coef    std err          t      P>|t|      [0.025      0.975]
--------------